In [1]:
# Importamos la libreria pandas
import pandas as pd

# 1. Cargar el dataset
data_monitor = pd.read_csv('files_folder/monitoreo_operaciones_2024.csv')

# 2. Convertir y pasar al índice
data_monitor = (
    data_monitor
    .assign(timestamp = lambda x: pd.to_datetime(x['timestamp']))
    .set_index('timestamp')
    .sort_index()
)

print("Indice temporal configurado. La tabla ahora es una serie de tiempo oficial")
print(data_monitor.head())

Indice temporal configurado. La tabla ahora es una serie de tiempo oficial
                    dispositivo servidor_id  latencia_ms  monto_transaccion
timestamp                                                                  
2024-01-01 00:15:17      Mobile      SRV-02       130.39              22.81
2024-01-01 01:08:58      Mobile      SRV-04       138.87               9.21
2024-01-01 01:47:44      Mobile      SRV-03       165.99             108.77
2024-01-01 05:27:55      Mobile      SRV-02       193.37              55.05
2024-01-01 06:00:50      Mobile      SRV-01       110.26              13.40


# Solución 1: El Cierre Contable (Resampling Mensual)

In [2]:
# Crear la variable que guardara la sumatoria de todos los ingresos por mes usando .sum y resample('M')
monto_mensual = (
    data_monitor['monto_transaccion']
    .resample('ME')
    .sum()
    .to_frame() # Convertimos a DataFrame para que se vea como tabla
)

print(" Reporte del monto total por mes")
print(monto_mensual.head())

 Reporte del monto total por mes
            monto_transaccion
timestamp                    
2024-01-31           83703.15
2024-02-29           78031.74
2024-03-31           82270.09
2024-04-30           82064.84
2024-05-31           81045.24


# Solución 2: Auditoría de Crisis (Slicing + GroupBy)

In [22]:
# Creamos una variable que guarde el promedio de latencia por día usando .groupby y .mean
latencia_diaria_agosto = (
    data_monitor.loc['2024-08']
    .groupby('servidor_id')['latencia_ms']
    .mean()
    .fillna(0)
    .reset_index(name='latencia_media_ms') # Convierte el resultado en un DataFrame con columnas servidor_id, timestamp y latencia_media_ms.
)



print("--- Reporte de la latencia por día en el transcurso de agosto---")
print(latencia_diaria_agosto)

--- Reporte de la latencia por día en el transcurso de agosto---
  servidor_id  latencia_media_ms
0      SRV-01         182.165747
1      SRV-02         184.316528
2      SRV-03         174.422074
3      SRV-04         183.621343


# Solución 3: Análisis de Tráfico de Dispositivos (Resampling Semanal)

In [13]:
trafico_semanal = pd.pivot_table(
    data_monitor,
    index=pd.Grouper(freq='W'),
    columns='dispositivo',
    aggfunc='size',
    fill_value=0
)
print("Número de Interacción por dispositivo a la semana")
print(trafico_semanal.head())

Número de Interacción por dispositivo a la semana
dispositivo  Desktop  Mobile  Tablet
timestamp                           
2024-01-07        49     124      20
2024-01-14        62     107      15
2024-01-21        58     118      27
2024-01-28        60     116      17
2024-02-04        57     120      21


# Solución 4: Crecimiento Month-over-Month (MoM)

In [17]:
data_month = (
    data_monitor
    .resample('ME')['monto_transaccion']
    .sum()
    .to_frame(name='total_ventas')
    .assign(
        # Generar columna con el ingreso del mes anterior
        ingreso_mes_anterior = lambda x: x['total_ventas'].shift(1),

        # Calcular crecimiento porcentual respecto al mes anterior
        crecimiento_pct = lambda x: (
            (x['total_ventas'] - x['ingreso_mes_anterior'])
            / x['ingreso_mes_anterior'] * 100
        ).fillna(0)
    )
)

print("----- Crecimiento mes por mes---")
print(data_month)

----- Crecimiento mes por mes---
            total_ventas  ingreso_mes_anterior  crecimiento_pct
timestamp                                                      
2024-01-31      83703.15                   NaN         0.000000
2024-02-29      78031.74              83703.15        -6.775623
2024-03-31      82270.09              78031.74         5.431572
2024-04-30      82064.84              82270.09        -0.249483
2024-05-31      81045.24              82064.84        -1.242432
2024-06-30      83871.70              81045.24         3.487509
2024-07-31      85472.52              83871.70         1.908653
2024-08-31      82955.24              85472.52        -2.945134
2024-09-30      91614.20              82955.24        10.438111
2024-10-31      80508.52              91614.20       -12.122226
2024-11-30      82936.55              80508.52         3.015867
2024-12-31      82675.40              82936.55        -0.314879
